# Lição 18: Protegendo Agentes de IA com Recibos Criptográficos

## Notebook Prático

Este notebook guia por quatro exercícios:

1. **Assinar seu primeiro recibo** para uma chamada de ferramenta do agente e verificá-lo.
2. **Manipular o recibo** e observar a falha na verificação.
3. **Construir uma cadeia de três recibos** e confirmar a integridade da cadeia.
4. **Envolver uma chamada de ferramenta do Microsoft Agent Framework** para que cada ação emita um recibo.

Todos os primitivos criptográficos são importados de bibliotecas bem mantidas (`pynacl` para Ed25519, `jcs` para JSON canônico RFC 8785, `hashlib` da biblioteca padrão do Python para SHA-256). A lógica do recibo em si é Python simples que você pode ler e modificar.

Execute as células na ordem. Cada seção é curta e autocontida.


## Configuração

Instale as duas dependências. Ambas possuem licenças permissivas (Apache-2.0 / MIT).


In [1]:
%pip install -q pynacl jcs

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## Utilitários Auxiliares

Estes dois auxiliares lidam com codificação base64url (sem preenchimento) e hash SHA-256 de objetos arbitrários. Eles mantêm o restante do caderno focado na lógica do recibo em si.


In [3]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Seção 1: Assine seu primeiro recibo

Imagine que nosso agente da **Contoso Travel** acabou de pesquisar voos de Sydney para Los Angeles para um cliente. Queremos registrar essa chamada de ferramenta como um recibo assinado para que um auditor futuro possa verificá-lo sem precisar confiar em nós.

### Passo 1.1: Gere uma chave de assinatura

Em produção, a chave de assinatura do agente ficaria em um módulo de segurança de hardware (HSM), Azure Key Vault ou uma loja protegida semelhante. Para esta lição, geramos uma chave nova em memória.


In [4]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

Public key (Ed25519, 32 bytes): g3SyD_ecOKa1L8RQ79-pDy9em81H-O_jzp9VG4a3EP0


### Etapa 1.2: Construir a carga útil do recibo

A carga útil contém tudo o que queremos que o recibo comprove: quem atuou, qual ferramenta, com quais argumentos, o que retornou, sob qual política e quando. Nós hashamos os argumentos e o resultado em vez de incluí-los inline para que o recibo não revele conteúdo sensível.


In [5]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-08-18T04:55:21Z",
  "sequence": 0,
  "previous_receipt_hash": null
}


### Passo 1.3: Assinar e montar o recibo

Três passos:

1. Canonicalizar o payload usando JCS para que duas implementações produzindo o mesmo recibo lógico gerem bytes idênticos.
2. Assinar os bytes canônicos diretamente com a chave privada Ed25519. PureEdDSA faz o hash da mensagem internamente, então um pré-hash extra modificaria o protocolo.

A assinatura é então anexada ao payload original para produzir o recibo final.


In [6]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    signature_bytes = signing_key.sign(canonical).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-08-18T04:55:21Z",
  "sequence": 0,
  "previous_receipt_hash": null,
  "signature": {
    "alg": "EdDSA",
    "sig": "Eu3MmrOSdGq2DuemkjNSPKfz5xbr18okNeTU11NJu2usnsnKtC-pjq2PZM1Oap-WXdVgYkL4iW6SeWPSZ2M9BQ",
    "public_key": "g3SyD_ecOKa1L8RQ79-pDy9em81H-O_jzp9VG4a3EP0"
  }
}


### Passo 1.4: Verificar o recibo

A verificação reverte o processo. Removemos a assinatura, recalculamos os bytes canônicos e verificamos a assinatura com a chave pública no recibo.

Um auditor realizando essa verificação não precisa de nada além do próprio recibo. Nenhum serviço para chamar, nenhum diretório de chaves para consultar, nenhuma confiança necessária.


In [7]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(canonical, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

# Regression control for the signature scope in draft revision 02. A receipt
# signed over SHA-256(JCS(payload)) is a signature over different bytes and
# must not verify as a direct-JCS Ed25519 receipt.
prehashed_signature = signing_key.sign(hashlib.sha256(canonicalize(payload)).digest()).signature
prehashed_receipt = {
    **receipt,
    "signature": {**receipt["signature"], "sig": b64url_nopad(prehashed_signature)},
}
print(f"Pre-hashed receipt valid: {verify_receipt(prehashed_receipt)}")

Receipt is valid: True
Pre-hashed receipt valid: False


Você deve ver `Receipt is valid: True` e `Pre-hashed receipt valid: False`. O caso positivo prova que o caminho de assinatura direto-JCS funciona; o controle negativo torna a regra do escopo da assinatura executável em vez de deixá-la como prosa.


## Seção 2: Alterar o recibo

O objetivo de recibos é que eles sejam evidentes de alterações. Vamos provar isso.

Vamos modificar exatamente um caractere do recibo e observar a falha na verificação.


In [8]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

Original policy_id:  contoso-travel-policy-v3
Tampered policy_id:  contoso-travel-policy-PERMISSIVE

Tampered receipt valid? False


### O que acabou de acontecer?

Quando alteramos `policy_id`, os bytes canônicos mudaram. A assinatura (que estava sobre os bytes canônicos originais) não corresponde mais. A verificação corretamente retorna `False`.

Não há como modificar qualquer campo do recibo e ainda assim tê-lo verificado, a menos que o atacante tenha a chave privada. Enquanto a chave privada estiver em um cofre de chaves e a chave pública for publicada, adulteração é impossível de esconder.

Experimente você mesmo: modifique o `tool_name` ou `agent_id` ou `timestamp` na célula acima e execute novamente. Cada alteração produz um recibo inválido.


## Seção 3: Encadeie os recibos juntos

Um único recibo protege uma ação. A maioria dos agentes realiza muitas ações. Para tornar toda a sequência evidenciável contra adulterações, vinculamos cada recibo ao anterior incluindo o hash do recibo anterior na carga útil do novo recibo.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Se alguém remover ou reordenar um recibo, a cadeia quebra exatamente naquele ponto. A verificação de qualquer recibo posterior falha porque seu `previous_receipt_hash` não corresponde mais ao hash real do seu predecessor.


In [9]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [10]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

Receipt 0: tool=lookup_flights, prev=None
Receipt 1: tool=hold_seat, prev=sha256:d54d1e138bb080dd6bd825d9f015efc74ad6c7f78ee9504d0b68b39d0733b39c
Receipt 2: tool=confirm_booking, prev=sha256:9fbe5dc2ffcffd7d167ef13a744aecd578e474fb536cb7104ddb1a359607f2f6


In [11]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): VALID
Receipt 2 (   confirm_booking): VALID


Agora quebre a cadeia adulterando o recibo do meio e revalide. O recibo adulterado falha na verificação da assinatura, E o próximo recibo falha na verificação do elo da cadeia (porque seu `previous_receipt_hash` não corresponde mais ao hash do recibo do meio modificado).


In [12]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): INVALID (failed: signature)
Receipt 2 (   confirm_booking): INVALID (failed: chain link)


O recibo 0 ainda verifica (não foi modificado e não tem predecessor do qual dependa). O recibo 1 falha na verificação da assinatura porque alteramos `tool_args_hash`. O recibo 2 falha na verificação do link da cadeia porque seu `previous_receipt_hash` foi calculado com base no recibo 1 original (agora modificado).

Mesmo que um atacante re-assine o recibo 1 modificado (o que ele não pode fazer sem a chave privada), a incompatibilidade do link da cadeia no recibo 2 ainda exporia a adulteração. Para ocultar a alteração, o atacante teria que re-assinar todos os recibos a partir do ponto da modificação, o que requer posse da chave privada.


## Seção 4: Envolver uma chamada de ferramenta do agente com assinatura de recibo

Em uma implantação real, você não quer que todo autor de agente se lembre de chamar `make_receipt`. Você quer que a assinatura do recibo seja automática para toda invocação de ferramenta.

Aqui está o padrão mais simples: uma classe wrapper que recebe qualquer função de ferramenta invocável e retorna uma versão que emite recibo. Isso se adapta a qualquer framework de agente, incluindo o Microsoft Agent Framework (`agent_framework.foundry`).

Se você não tem um projeto Microsoft Foundry configurado, o mock local abaixo ainda demonstra o padrão.


In [13]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [14]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


Tool was called 3 times.
Each call produced a signed receipt linked to the previous one.

Receipt 0 (lookup_flights): VALID
Receipt 1 (lookup_flights): VALID
Receipt 2 (lookup_flights): VALID


### Integração com o Microsoft Agent Framework

O wrapper `ReceiptedTool` acima é independente de framework. Para usá-lo dentro de um agente criado com o Microsoft Agent Framework, registre a função encapsulada como uma ferramenta. Um esboço (você substituiria o mock por um registro real de ferramenta no Microsoft Foundry):

```python
# Pseudocódigo mostrando a forma de integração.
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="Você é um agente de viagem da Contoso ...",
#     tools=[receipted_lookup],   # a ferramenta encapsulada, não a função bruta
# )
# response = agent.run("Encontre voos de Sydney para Los Angeles em junho.")
#
# # Após a execução, cada chamada de ferramenta feita pelo agente tem um recibo assinado:
# audit_chain = receipted_lookup.receipts
```

O framework do agente não precisa saber nada sobre recibos. A assinatura do recibo é encapsulada em torno da ferramenta, não integrada ao framework. É assim que você adiciona procedência ao código do agente existente sem reescrever o agente.


## Recapitulação e desafio adicional

Você:

- Gerou um par de chaves Ed25519.
- Construiu e assinou um recibo para uma chamada de ferramenta do agente.
- Verificou o recibo offline usando apenas a chave pública.
- Alterou um recibo e observou a falha na verificação.
- Construiu uma sequência encadeada por hash de três recibos.
- Alterou o meio da cadeia e observou falhas tanto na assinatura quanto no elo da cadeia.
- Envolveu uma função de ferramenta com assinatura automática de recibo.

**Desafio adicional.** Estenda o esquema do recibo com um campo `request_id` (um UUID para rastreamento distribuído). Atualize `make_receipt` para incluí-lo e confirme que os recibos ainda verificam de ponta a ponta. Depois modifique o campo após a assinatura e confirme que a verificação falha. Isso força você a internalizar como cada byte da codificação canônica contribui para a assinatura.

**Limite importante.** Os recibos provam três coisas e somente três coisas: atribuição (esta chave assinou este conteúdo), integridade (o conteúdo não mudou desde a assinatura) e ordenação (este recibo veio depois daquele recibo). Eles NÃO provam que a ação do agente foi correta, que a política nomeada em `policy_id` foi realmente avaliada ou que o agente seguiu todas as regras. Recibos são uma base. Governança é o sistema que você constrói em cima.

Leia o README da lição novamente com esse limite em mente. O erro mais comum que as equipes cometem com recibos é assumir que "temos recibos" significa "estamos governados". Não significa. Recibos tornam o comportamento do agente auditável. Eles não o tornam correto.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:
Este documento foi traduzido usando o serviço de tradução por IA [Co-op Translator](https://github.com/Azure/co-op-translator). Embora nos esforcemos pela precisão, por favor, esteja ciente de que traduções automatizadas podem conter erros ou imprecisões. O documento original em seu idioma nativo deve ser considerado a fonte autorizada. Para informações críticas, recomenda-se tradução profissional humana. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações incorretas decorrentes do uso desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
